# Hierarchischer Ansatz: HistGradientBoosting

**Autor:** jd · **Datum:** 2026-10-12

**Ziel:** HistGradientBoosting je Stufe im hierarchischen Ansatz testen (#103).

In [ ]:
%load_ext autoreload
%autoreload 2

from sklearn.ensemble import HistGradientBoostingClassifier

from awp2.config import MODEL_DOCS_FIGURES_DIR, SEED
from awp2.data import load_split
from awp2.evaluation import compare_predictions
from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import HierarchicalClassifier, summarize_hierarchical, summary_table
from awp2.models.tuned import refit_untracked, tuned_hierarchical_svm
from awp2.plots import plot_error_breakdown, plot_tuning_candidates, save_doc_figure
from awp2.preprocessing import PreprocessingConfig

In [ ]:
split = load_split()
split.X_train.shape

## Hypothese

Gradient Boosting baut Bäume nacheinander auf die Fehler der vorigen und ist bei tabellarischen Daten oft stark. Im kombinierten Ansatz erreichte HistGradientBoosting 0.777 (`combined_hist_gradient_boosting`). Frage: Profitiert es von der Zerlegung in Kultur und Stadium? Ausgeglichene Gewichte über `balance_samples=True` (je Stufe umskaliert). Ein Fit dauert rund 30 s, daher nur die Lernrate je Stufe bei Standard-Baumgröße.

Gleicher Ablauf wie bei der Referenz `hierarchical_svm` (#87): CV-Suche mit `tune()` auf dem
Trainingsteil, dann genau ein `run()`. Verglichen wird per CV mit der SVM-Referenz, danach
Fehlerzerlegung, Oracle-Kultur und McNemar gegen die SVM auf der Validierung.

In [ ]:
reference = tuned_hierarchical_svm()
svm_cv_score = reference.cv_score
preprocessing = PreprocessingConfig(use_spectral_standard_scale=True)
TOP_CANDIDATES = 15

model = HierarchicalClassifier(
    HistGradientBoostingClassifier(random_state=SEED),
    HistGradientBoostingClassifier(random_state=SEED),
)

learning_rate = [0.05, 0.1]
param_grid = {
    "crop_estimator__learning_rate": learning_rate,
    "stage_estimator__learning_rate": learning_rate,
}

## CV-Suche

In [ ]:
tuned = tune(
    model,
    TuneConfig(
        name="hierarchical_hgb_search",
        approach="hierarchical",
        description="HistGradientBoosting je Stufe im hierarchischen Ansatz (#103).",
        preprocessing=preprocessing,
        balance_samples=True,
        param_grid=param_grid,
    ),
)
tuned.best_params, round(tuned.best_score, 4), round(tuned.best_score - svm_cv_score, 4)

In [ ]:
tuning_ax = plot_tuning_candidates(
    tuned.candidates.nsmallest(TOP_CANDIDATES, "rank"), color_by="crop_estimator__learning_rate"
)
tuning_ax.axvline(svm_cv_score, color="black", linestyle="--", linewidth=1)
tuning_ax.set_title("Hierarchisch HistGradientBoosting: CV-Kandidaten (gestrichelt: SVM-Referenz)")
save_doc_figure(tuning_ax.figure, "hierarchical_hgb_tuning", MODEL_DOCS_FIGURES_DIR)

## Einmalige Bewertung

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name="hierarchical_hgb",
        approach="hierarchical",
        description="Bester HistGradientBoosting aus hierarchical_hgb_search, einmal bewertet.",
        preprocessing=preprocessing,
        balance_samples=True,
        tuning_run=tuned.run_id,
    ),
)
result.metrics.model_dump(), result.run_id

## Vergleich mit der SVM-Referenz

Die SVM-Referenz (`tuned_hierarchical_svm()`) wird ohne MLflow nachtrainiert
(`refit_untracked`), um die Vorhersagen Zeile für Zeile zu vergleichen.

In [ ]:
svm_result = refit_untracked(reference, "hierarchical_svm_refit")
summaries = {
    "RBF-SVM (Referenz)": summarize_hierarchical(svm_result.pipeline, split.X_val, split.y_val),
    "HistGradientBoosting": summarize_hierarchical(result.pipeline, split.X_val, split.y_val),
}
breakdowns = {label: summary.breakdown for label, summary in summaries.items()}
(
    summary_table(summaries).round(3),
    compare_predictions(split.y_val, svm_result.y_pred, result.y_pred),
)

In [ ]:
breakdown_ax = plot_error_breakdown(breakdowns)
breakdown_ax.set_title("Fehlerzerlegung: HistGradientBoosting vs. RBF-SVM")
save_doc_figure(breakdown_ax.figure, "hierarchical_hgb_breakdown", MODEL_DOCS_FIGURES_DIR)

## Befund

- CV 0.783, Validierung 0.795 – in der CV gleichauf mit dem kombinierten HistGradientBoosting
  (0.781); der Vorsprung auf der Validierung ist eine einzelne Zahl.
- Kleinere Lernrate (0.05, Rasterrand) gewinnt auf beiden Stufen.
- Abstand zur SVM signifikant (McNemar p ≈ 0.001), vor allem bei der Kultur (11 % vs. 9 %).